# Reach-Avoid (2D)

**What you will learn:** define target and constraint level sets, call `impl.reach(l, g)`,
read a backward reachable tube, and visualize it in 2D and 3D.

**pyspect API:** `TVHJImpl`, `reach`

A boat must reach a target while staying outside two obstacles, despite a current.
State `(x, y)`, computed with `ReachAvoid2D`.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import ReachAvoid2D

In [ ]:
# Same grid as hjr_examples: x in [-10,10], y in [0,10].
# The horizon is given as positive time: `reach` integrates backwards internally.
AXES = [
    dict(name='t', bounds=[0, 10], points=20),
    dict(name='x', bounds=[-10, 10], points=50),
    dict(name='y', bounds=[0, 10], points=50),
]

impl = TVHJImpl(dict(cls=ReachAvoid2D), AXES, accuracy='very_high')

In [ ]:
X = impl.grid.states[..., 0]
Y = impl.grid.states[..., 1]

# Formulas identical to hjr_examples, which uses the "positive = inside the set" convention
l1 = 1 - jnp.abs(X - 0.0)
l2 = 1 - 5 * jnp.abs(Y - 1.0)
target_pos = jnp.maximum(jnp.minimum(l1, l2), -5.0)

g1 = jnp.abs(X + 5.0) - 2
g2 = jnp.abs(X - 5.0) - 2
g3 = jnp.abs(Y - 2.5) - 2.5
constraint_pos = jnp.minimum(jnp.maximum(g1, g3), jnp.maximum(g2, g3))

# pyspect uses the opposite convention ("negative = inside the set")
l = -target_pos
g = -constraint_pos

In [ ]:
V = impl.reach(l, g)
print('shape:', V.shape)  # (nt, nx, ny)

# V[-1] is the target (zero horizon), V[0] is the tube over the full horizon.
# `ttr[i]` is the time-to-go associated with V[i].
ttr = np.array(impl.timeline)[::-1]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(
    np.array(V[0]).T,
    origin='lower',
    extent=[-10, 10, 0, 10],
    aspect='auto',
    cmap='viridis',
)
ax.contour(X.T, Y.T, np.array(V[0]).T, levels=[0], colors='black', linewidths=1.0)
ax.contour(X.T, Y.T, np.array(l).T, levels=[0], colors='green', linewidths=0.8)
ax.contour(X.T, Y.T, np.array(g).T, levels=[0], colors='red', linewidths=0.8)
plt.colorbar(im, ax=ax, label=r'$V(x,t)$')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Tube over {ttr[0]:.1f} s (target in green, obstacles in red)')
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

V_np = np.array(V)
fig, ax = plt.subplots(figsize=(7, 4))

def update(i):
    ax.clear()
    ax.imshow(
        V_np[i].T,
        origin='lower',
        extent=[-10, 10, 0, 10],
        aspect='auto',
        cmap='viridis',
    )
    ax.contour(X.T, Y.T, V_np[i].T, levels=[0], colors='black', linewidth=0.7)
    ax.contour(X.T, Y.T, np.array(g).T, levels=[0], colors='red', linewidths=0.8)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'time-to-go = {ttr[i]:.1f} s')

frames = list(range(len(V_np) - 1, -1, -1))
ani = FuncAnimation(fig, update, frames=frames, interval=200, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


## The value function as a 3D surface

Same view as the original presentation: `V`, the obstacles `g` and the target `l` are
drawn as surfaces over the `(x, y)` plane. Everything is clipped to ±5, otherwise the
obstacle walls at the domain edges flatten the picture. The saturated plateaus of `l`
and `g` are masked so they do not hide `V`.

The reach-avoid set is wherever the blue surface dips below zero.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.lines import Line2D

x = np.array(impl.grid.coordinate_vectors[0])
y = np.array(impl.grid.coordinate_vectors[1])
Xg, Yg = np.meshgrid(x, y, indexing='ij')

CLIP = 5.0
# Hide the saturated plateaus so they do not occlude the value function
mask = lambda A: np.where(np.abs(np.asarray(A)) < CLIP, np.asarray(A), np.nan)
ls, gs = mask(l), mask(g)
Vs0 = np.clip(V_np[0], -CLIP, CLIP)

fig = plt.figure(figsize=(9, 6.5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(Xg, Yg, Vs0, rstride=1, cstride=1,
                linewidth=0, antialiased=True, alpha=0.9, color='lightblue')
ax.plot_surface(Xg, Yg, gs, rstride=1, cstride=1,
                linewidth=0.2, alpha=0.45, color='red')
ax.plot_surface(Xg, Yg, ls, rstride=1, cstride=1,
                linewidth=0.2, alpha=0.45, color='green')
ax.view_init(elev=28, azim=-55)
ax.invert_xaxis()
ax.set_zlim(-CLIP, CLIP)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.set_zlabel('$V$')
ax.set_title(f'Value function at full horizon ({ttr[0]:.1f} s)')
ax.legend(handles=[Line2D([0], [0], color='lightblue', lw=6, label=r'$V(x,y,t)$'),
                   Line2D([0], [0], color='red', lw=6, label=r'$g(x,y)$'),
                   Line2D([0], [0], color='green', lw=6, label=r'$\ell(x,y)$')],
          loc='upper left')
plt.tight_layout()
plt.show()

## The reachable tube as a Plotly isosurface

Zero level set of `V` in `(x, y, t)` space: every point under the surface is a state
from which the target can still be reached without hitting an obstacle, given the
remaining time budget.


In [ ]:
# Default axes of transform_to_isosurface put time on z: (x, y, t)
fig = impl.plot(V, method='isosurface', level=0.0, axes=('x', 'y', 't'),
                colorscale='Blues', opacity=0.85,
                name='reachable tube  V = 0')
fig.update_layout(
    title='Reach-avoid tube in (x, y, t)',
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='time-to-go',
        aspectmode='manual',
        aspectratio=dict(x=1.4, y=0.7, z=0.7),
    ),
    width=800, height=560,
)
fig.show()
